# SI — single ATTO 647N molecules on two other colour cameras (Thorlabs CS505CU, ZWO ASI 585MC)

**Makes up:** four SI figures, 100 pM ATTO 647N in 1 % PVA:

- `ATTO647N_Thorcam` (Thorlabs CS505CU): a) one field of view, b) photobleaching traces of the three
  boxed molecules;
- `ATTO647N_Thorlabs_ZoomIns` (Thorlabs CS505CU): 25 PSFs;
- `ZWO_ATTO647N` (ZWO ASI 585MC): as `ATTO647N_Thorcam`;
- `ATTO647N_ZWO_ZoomIns` (ZWO ASI 585MC): 25 PSFs.

Everything is in photoelectrons: the images have photoelectron colour bars, and each zoom-in has its
own colour bar and the 2 × 2 Bayer key of its top-left corner (the colour filters under those pixels).

**Data:** BioStudies `S-BIAD3210`, `2026_Multicolour_Paper/`:
- `ThorLabs/20250306_MoreDyes/data/atto647n_100pM_PVA_40mW/ATTO647n_100pM_PVA_40mW_2025-03-06T13-34-22.325_0.tif`
  (637 frames, 100 ms, ~450 MB). The stage was moved during the recording; frames 8–627 are its
  first field of view.
- `ZWO/20260402_ATTO647N_ZWO/ATTO647N_in_PVA_100pM/QuadBand_635LP_10mW_both_manualShutter_100ms_3/`,
  position `Pos-5-000_000` (1,000 frames, 100 ms, ~560 MB) and its MicroManager metadata.

**Photoelectrons.** ZWO: the camera's calibration (gain, offset and relative QE per pixel), cropped
to the acquisition's ROI. Thorlabs: the acquisition was recorded at gain 200 (20 dB) and black level
0, not the settings of the CS505CU calibration in `Camera_Calibrations/` (offset ~100 ADU), so its
gain is measured from the recording itself: the photon transfer (temporal variance against mean) of
each pixel over the bleached end of the first field of view, per Bayer pixel type, with the offset
at 0 (black level 0). The analogue gain is the same for every pixel type; the fits of the pixel
types that are never clipped at zero are averaged.

**Molecules.** The field of view, boxes and trace windows are those of the published figures
(found again by matching the published images to the data). The 25 zoom-ins per camera are the
brightest clear single PSFs in that frame: isolated (no other peak above 20 % of theirs within the
crop), skipping the brightest 2 % (likely more than one molecule). Each trace is the box's summed
photoelectrons minus the local background (mean of a 4 px frame around the box) per frame.

---
## 0 — Imports

In [ ]:
import sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

# _biostudies_download.py and _atto647n_cameras.py live in the parent folder (notebooks/figures/).
sys.path.insert(0, str(Path.cwd().resolve().parent))
import _biostudies_download as bd
import _atto647n_cameras as cams

from pyS3M import IOFunctions
from pyS3M import HelperFunctions
from pyS3M import PlottingBase
from pyS3M.CameraDefaults import get_camera_config

IO = IOFunctions.IO_Functions()
H_F = HelperFunctions.Helper_Functions()
plotter = PlottingBase.PublicationPlotter(dark_background=False)

print("Python:", sys.version)

---
## 1 — Configuration

In [ ]:
REPO_ROOT = Path.cwd().resolve().parents[2]   # notebooks/figures/SI -> repo root
CALIB_DIR = REPO_ROOT / "Camera_Calibrations"
OUT_ROOT  = Path("outputs") / "FigureSI_ATTO647N_Cameras"   # git-ignored (bare outputs/ rule)
FIG_DIR   = OUT_ROOT / "figures"
DATA_DIR  = OUT_ROOT / "data"   # point at a drive with ~1 GB free
FIG_DIR.mkdir(parents=True, exist_ok=True)
DATA_DIR.mkdir(parents=True, exist_ok=True)

HALF_ROI = 8           # zoom-ins are 16 x 16 px
N_ZOOM = 25
EXPOSURE_S = 0.1       # both cameras: 100 ms frames

# Thorlabs CS505CU. Its camera settings file (not deposited) gives the ROI, gain and black level.
THOR_FOLDER = "2026_Multicolour_Paper/ThorLabs/20250306_MoreDyes/data/atto647n_100pM_PVA_40mW"
THOR_FILE = "ATTO647n_100pM_PVA_40mW_2025-03-06T13-34-22.325_0.tif"
THOR_ROI = (880, 884, 696, 512)          # x0, y0, width, height on the sensor
THOR_PIXEL_NM = 69.0                     # same 3.45 um pixels and optics as the Ximea
THOR_FOV = (8, 628)                      # frames of the first field of view (the stage moves after)
THOR_FRAME = 13                          # frame shown, and the first of the traces
THOR_BOXES = [(102, 140, 15), (201, 230, 15), (410, 412, 15)]   # (col, row, size), red/lime/light blue
THOR_PTC_FRAMES = (500, 628)             # bleached end of the field of view, for the gain

# ZWO ASI 585MC
ZWO_FOLDER = ("2026_Multicolour_Paper/ZWO/20260402_ATTO647N_ZWO/ATTO647N_in_PVA_100pM/"
              "QuadBand_635LP_10mW_both_manualShutter_100ms_3")
ZWO_STEM = "QuadBand_635LP_10mW_both_manualShutter_100ms_3_MMStack_Pos-5-000_000"
ZWO_FRAME = 10                           # frame shown, and the first of the traces
ZWO_TRACE_FRAMES = 300                   # 30 s
ZWO_BOXES = [(75, 130, 15), (205, 220, 25), (163, 415, 15)]

print(f"Data dir   : {DATA_DIR.resolve()}")
print(f"Figures dir: {FIG_DIR.resolve()}")

---
## 2 — Download

Two files and the ZWO position's metadata (~1 GB). Re-running skips complete downloads.

In [ ]:
thor_tif = bd.download_file(f"{THOR_FOLDER}/{THOR_FILE}", DATA_DIR / "Thorlabs")
zwo = bd.download_acquisition(ZWO_FOLDER, DATA_DIR / "ZWO", stem=ZWO_STEM)
zwo_tif, zwo_meta = zwo["images"][0], zwo["metadata"][0]
print(thor_tif)
print(zwo_tif)

---
## 3 — Thorlabs CS505CU: gain from the recording

Photon transfer per Bayer pixel type (`row % 2, col % 2` of the sensor) over the bleached end of
the first field of view. Pixels that ever read ≤ 1 ADU are left out (their noise is clipped at the
sensor's zero, black level 0); a pixel type with none left is not fitted.

In [ ]:
ptc_stack = IO.read_tiff(thor_tif, dtype="float32", frame=list(range(*THOR_PTC_FRAMES)))
ptc = cams.photon_transfer_gain(ptc_stack)
del ptc_stack
for parity, (g, rn, used) in ptc.items():
    print(f"sensor parity {parity}: gain {g:.2f} ADU/pe, read noise {rn:.2f} pe ({used:.0%} of pixels unclipped)")
THOR_GAIN = float(np.nanmean([g for g, _, _ in ptc.values()]))
print(f"gain used: {THOR_GAIN:.2f} ADU per photoelectron, offset 0 ADU")


def thor_to_pe(raw):
    return np.asarray(raw, dtype=np.float32) / THOR_GAIN

---
## 4 — ZWO ASI 585MC: calibration

In [ ]:
x0_zwo, y0_zwo, w_zwo, h_zwo = IO.metadata_reader_imageJ(zwo_meta)
zwo_cal = H_F.crop_calibration_maps(
    {k: IO.read_tiff(CALIB_DIR / "ZWO_Camera" / f"{k}.tif") for k in ("gain", "offset", "rqe")},
    x0_zwo, y0_zwo, w_zwo, h_zwo)
zwo_cfg = get_camera_config("zwo")
print(f"ZWO ROI: x0={x0_zwo}, y0={y0_zwo}, {w_zwo} x {h_zwo}; pixel {zwo_cfg.pixel_size * 1e3:.1f} nm")


def zwo_to_pe(raw):
    return IO.convert_to_photoelectrons(np.asarray(raw, dtype=np.float32), gain_map=zwo_cal["gain"],
                                        offset_map=zwo_cal["offset"], rqe=zwo_cal["rqe"])

---
## 5 — Fields of view and photobleaching traces (`ATTO647N_Thorcam`, `ZWO_ATTO647N`)

Traces are read 100 frames at a time.

In [ ]:
def traces(tif, first, last, to_pe, boxes, chunk=100):
    out = [[] for _ in boxes]
    for start in range(first, last, chunk):
        pe = to_pe(IO.read_tiff(tif, dtype="float32", frame=list(range(start, min(start + chunk, last)))))
        for k, box in enumerate(boxes):
            out[k].append(cams.box_trace(pe, box))
    return [np.concatenate(t) for t in out]


thor_frame = thor_to_pe(IO.read_tiff(thor_tif, dtype="float32", frame=THOR_FRAME))
thor_traces = traces(thor_tif, THOR_FRAME, THOR_FOV[1], thor_to_pe, THOR_BOXES)
fig, axs = cams.plot_fov_traces(plotter, thor_frame, THOR_BOXES, thor_traces, EXPOSURE_S, THOR_PIXEL_NM)
plotter.save_or_show(fig, save_path=str(FIG_DIR / "ATTO647N_Thorcam.svg"), show=True)

zwo_frame = zwo_to_pe(IO.read_tiff(zwo_tif, dtype="float32", frame=ZWO_FRAME))
zwo_traces = traces(zwo_tif, ZWO_FRAME, ZWO_FRAME + ZWO_TRACE_FRAMES, zwo_to_pe, ZWO_BOXES)
fig, axs = cams.plot_fov_traces(plotter, zwo_frame, ZWO_BOXES, zwo_traces, EXPOSURE_S,
                                zwo_cfg.pixel_size * 1e3)
plotter.save_or_show(fig, save_path=str(FIG_DIR / "ZWO_ATTO647N.svg"), show=True)

---
## 6 — PSF zoom-ins (`ATTO647N_Thorlabs_ZoomIns`, `ATTO647N_ZWO_ZoomIns`)

25 clear single PSFs per camera from the frame above, brightest first. The Bayer key comes from each
crop's position on the sensor and the camera's mosaic, read off the red dye's own signal (R > G > B
at ATTO 647N's ~670 nm emission); for the ZWO it must agree with `CameraDefaults`.

In [ ]:
def zoom_ins(frame_pe, roi_origin):
    centres = cams.pick_psfs(frame_pe, N_ZOOM, HALF_ROI)
    crops = [frame_pe[r - HALF_ROI:r + HALF_ROI, c - HALF_ROI:c + HALF_ROI] for r, c in centres]
    origins = [(roi_origin[0] + r - HALF_ROI, roi_origin[1] + c - HALF_ROI) for r, c in centres]   # on the sensor
    return centres, crops, origins


thor_centres, thor_crops, thor_origins = zoom_ins(thor_frame, (THOR_ROI[1], THOR_ROI[0]))
thor_unit, thor_frac = cams.bayer_from_red_dye(thor_crops, thor_origins)
print("Thorlabs mosaic from the red dye:", thor_unit.tolist(), "signal per parity:", np.round(thor_frac, 3).tolist())
fig, axs = cams.plot_zoom_ins(plotter, thor_crops,
                              [cams.pixel_types(thor_unit, o, c.shape) for o, c in zip(thor_origins, thor_crops)],
                              THOR_PIXEL_NM)
plotter.save_or_show(fig, save_path=str(FIG_DIR / "ATTO647N_Thorlabs_ZoomIns.svg"), show=True)

zwo_centres, zwo_crops, zwo_origins = zoom_ins(zwo_frame, (y0_zwo, x0_zwo))
zwo_unit, zwo_frac = cams.bayer_from_red_dye(zwo_crops, zwo_origins)
print("ZWO mosaic from the red dye:", zwo_unit.tolist(), " CameraDefaults:", zwo_cfg.mosaic_unit.tolist())
if not np.array_equal(zwo_unit, zwo_cfg.mosaic_unit):
    raise ValueError("the red dye's signal disagrees with the ZWO mosaic in CameraDefaults")
fig, axs = cams.plot_zoom_ins(plotter, zwo_crops,
                              [cams.pixel_types(zwo_unit, o, c.shape) for o, c in zip(zwo_origins, zwo_crops)],
                              zwo_cfg.pixel_size * 1e3)
plotter.save_or_show(fig, save_path=str(FIG_DIR / "ATTO647N_ZWO_ZoomIns.svg"), show=True)

print("Thorlabs zoom-in centres (row, col):", thor_centres.tolist())
print("ZWO zoom-in centres (row, col):", zwo_centres.tolist())